# Laboratorio 7 — MM3014 Teoría de Probabilidades
## Simulación de Monte Carlo: Álbum Panini FIFA 2026

**Parámetros reales del álbum:**
- N = 980 estampas diferentes
- S = 7 estampas por sobre (todas distintas dentro del mismo sobre)
- Precio sobre individual: Q 9.50
- Precio caja (104 sobres): Q 975.00
- Semilla: 2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display
import pandas as pd

# Configuración de estilo
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

---
## Etapa 1: Simulación básica con álbum reducido

### Parámetros
| Parámetro | Valor |
|-----------|-------|
| N (estampas distintas) | 100 |
| S (estampas por sobre) | 7 |
| R (simulaciones) | 10 000 |
| Semilla | 2026 |

In [ ]:
# ── Parámetros Etapa 1 ──────────────────────────────────────────────────────
N = 100          # estampas diferentes
S = 7            # estampas por sobre
R = 10_000       # número de simulaciones
SEED = 2026

rng = np.random.default_rng(SEED)

sobres_sim   = np.empty(R, dtype=int)   # sobres necesarios por simulación
repetidas_sim = np.empty(R, dtype=int)  # repetidas acumuladas por simulación

for i in range(R):
    album      = np.zeros(N, dtype=bool)  # False = no obtenida
    n_sobres   = 0
    n_repetidas = 0

    while album.sum() < N:
        # S estampas distintas dentro del mismo sobre
        sobre = rng.choice(N, size=S, replace=False)
        n_sobres += 1
        for sticker in sobre:
            if album[sticker]:
                n_repetidas += 1
            else:
                album[sticker] = True

    sobres_sim[i]    = n_sobres
    repetidas_sim[i] = n_repetidas

print("Simulación completa.")

### Resultados estadísticos

In [ ]:
media_sobres   = sobres_sim.mean()
std_sobres     = sobres_sim.std()
media_rep      = repetidas_sim.mean()
std_rep        = repetidas_sim.std()
prob_mas30     = (sobres_sim > 30).mean()

print(f"=== Etapa 1: Álbum reducido (N={N}, S={S}, R={R:,}) ===")
print()
print(f"Número de sobres necesarios")
print(f"  Media            : {media_sobres:.4f}")
print(f"  Desv. estándar   : {std_sobres:.4f}")
print()
print(f"Estampas repetidas")
print(f"  Media            : {media_rep:.4f}")
print(f"  Desv. estándar   : {std_rep:.4f}")
print()
print(f"P(sobres > 30)     : {prob_mas30:.4f}  ({prob_mas30*100:.2f}%)")

### Visualización: histograma de sobres necesarios

In [ ]:
# Mínimo teórico: ceil(N / S)
min_teorico = int(np.ceil(N / S))   # = 15 sobres si no hubiera repetidas

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(sobres_sim, bins=50, color='steelblue', edgecolor='white', alpha=0.85,
        label='Simulaciones')
ax.axvline(media_sobres,  color='crimson',   lw=2, ls='-',  label=f'Media muestral = {media_sobres:.2f}')
ax.axvline(min_teorico,   color='darkorange', lw=2, ls='--', label=f'Mínimo teórico = {min_teorico}')

ax.set_xlabel('Número de sobres', fontsize=12)
ax.set_ylabel('Frecuencia', fontsize=12)
ax.set_title(f'Distribución del número de sobres necesarios\n(N={N}, S={S}, R={R:,})', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('etapa1_histograma.png', bbox_inches='tight')
plt.show()
print(f"Figura guardada como 'etapa1_histograma.png'")

---
## Preguntas de análisis — Etapa 1

### Pregunta 1: Número mínimo de sobres si no hubiera repetidas

In [ ]:
# Si cada sobre trae S=7 estampas DISTINTAS y todas son nuevas,
# el mínimo de sobres sería ceil(N/S).
min_sobres = int(np.ceil(N / S))
print(f"Número mínimo de sobres (sin repetidas): ceil({N}/{S}) = ceil({N/S:.4f}) = {min_sobres}")
print()
# ¿Ocurrió en la simulación?
ocurrencias_min = (sobres_sim == min_sobres).sum()
print(f"Simulaciones con exactamente {min_sobres} sobres: {ocurrencias_min} de {R:,}")
print(f"  → Probabilidad empírica: {ocurrencias_min/R:.6f}")
print()
print("Interpretación:")
print(f"  El mínimo absoluto es {min_sobres} sobres. En la práctica es casi imposible")
print(f"  porque requeriría que ninguna estampa se repita entre sobres distintos.")

### Pregunta 2: Valor esperado teórico (Teoría del Coleccionista)

In [ ]:
gamma = 0.5772156649  # constante de Euler-Mascheroni

# H_N = sum(1/k, k=1..N)  — valor exacto por suma
H_N_exacto = sum(1/k for k in range(1, N+1))
# Aproximación: ln(N) + gamma
H_N_aprox  = np.log(N) + gamma

E_T_exacto = (N / S) * H_N_exacto
E_T_aprox  = (N / S) * H_N_aprox

print(f"H_{N} (suma exacta)      = {H_N_exacto:.6f}")
print(f"H_{N} (ln({N}) + γ)      = {H_N_aprox:.6f}")
print()
print(f"E[sobres] = (N/S)·H_N = ({N}/{S})·H_N")
print(f"  Valor teórico exacto  : {E_T_exacto:.4f} sobres")
print(f"  Valor teórico aprox   : {E_T_aprox:.4f} sobres")
print(f"  Media simulada        : {media_sobres:.4f} sobres")
print()
error_pct = abs(media_sobres - E_T_exacto) / E_T_exacto * 100
print(f"  Error relativo simulación vs teoría: {error_pct:.2f}%")

### Pregunta 3: Valor esperado teórico de estampas repetidas

In [ ]:
# Estampas totales obtenidas = S * E[sobres]; estampas nuevas = N
# → E[repetidas] = S * E[sobres] - N
E_rep_exacto = S * E_T_exacto - N
E_rep_aprox  = S * E_T_aprox  - N

print(f"E[repetidas] = S·E[sobres] - N")
print(f"  Valor teórico exacto  : {E_rep_exacto:.4f} estampas")
print(f"  Valor teórico aprox   : {E_rep_aprox:.4f} estampas")
print(f"  Media simulada        : {media_rep:.4f} estampas")
print()
error_rep = abs(media_rep - E_rep_exacto) / E_rep_exacto * 100
print(f"  Error relativo simulación vs teoría: {error_rep:.2f}%")

### Pregunta 4: Interpretación de la desviación estándar

In [ ]:
cv = std_sobres / media_sobres  # coeficiente de variación
print(f"Media de sobres        : {media_sobres:.4f}")
print(f"Desv. estándar sobres  : {std_sobres:.4f}")
print(f"Coeficiente de variación: {cv:.4f}  ({cv*100:.2f}%)")
print()
print("Interpretación:")
print(f"  La desv. estándar es ~{cv*100:.1f}% de la media, lo cual indica una variabilidad")
print(f"  considerable. Esto se debe a la aleatoriedad del proceso de coleccionar:")
print(f"  al principio casi todo sobre trae estampas nuevas (variabilidad baja),")
print(f"  pero al final se necesitan muchos sobres para obtener las últimas estampas")
print(f"  faltantes, lo que genera una cola derecha larga en la distribución.")
print(f"  P(sobres > 30) = {prob_mas30:.4f} confirma que más del umbral teórico mínimo")
print(f"  es común en la práctica.")

---
## Etapa 2: Análisis de la probabilidad de éxito en función del número de sobres

Para cada valor de M en {20, 25, 30, 35, 40, 45, 50, 60, 70, 80}, se compran exactamente M sobres
y se registra si el álbum quedó completo.

In [ ]:
M_values = [20, 25, 30, 35, 40, 45, 50, 60, 70, 80]
R2       = 10_000
rng2     = np.random.default_rng(SEED)

prob_exito = {}

for M in M_values:
    exitos = 0
    for _ in range(R2):
        album = np.zeros(N, dtype=bool)
        for _ in range(M):
            sobre = rng2.choice(N, size=S, replace=False)
            album[sobre] = True
        if album.all():
            exitos += 1
    prob_exito[M] = exitos / R2

print(f"{'M':>5}  {'P(completar)':>14}")
print("-" * 22)
for M, p in prob_exito.items():
    print(f"{M:>5}  {p:>14.4f}")

### Visualización: gráfica de barras de probabilidad de éxito

In [ ]:
M_arr = list(prob_exito.keys())
P_arr = list(prob_exito.values())

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(M_arr, P_arr, color='steelblue', edgecolor='white', alpha=0.85, width=3.5)

# Etiquetas encima de cada barra
for bar, p in zip(bars, P_arr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{p:.3f}', ha='center', va='bottom', fontsize=9)

ax.axhline(0.5, color='crimson',    lw=2, ls='--', label='P = 0.50')
ax.axhline(0.9, color='darkorange', lw=2, ls=':',  label='P = 0.90')

ax.set_xlabel('Número de sobres comprados (M)', fontsize=12)
ax.set_ylabel('P(álbum completo)', fontsize=12)
ax.set_title(f'Probabilidad de completar el álbum según sobres comprados\n(N={N}, S={S}, R={R2:,})', fontsize=13)
ax.set_ylim(0, 1.08)
ax.set_xticks(M_arr)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('etapa2_probabilidad.png', bbox_inches='tight')
plt.show()
print("Figura guardada como 'etapa2_probabilidad.png'")

---
## Preguntas de análisis — Etapa 2

### Pregunta 1: M donde la probabilidad supera por primera vez el 50% y el 90%

In [ ]:
M50 = next((M for M, p in prob_exito.items() if p >= 0.50), None)
M90 = next((M for M, p in prob_exito.items() if p >= 0.90), None)

print(f"Primer M con P(completar) ≥ 50%: M = {M50}  (P = {prob_exito[M50]:.4f})")
print(f"Primer M con P(completar) ≥ 90%: M = {M90}  (P = {prob_exito[M90]:.4f} {'(si existe en rango)' if M90 else ''})")

### Pregunta 2: Comparación del M al 50% con la mediana de la Etapa 1

In [ ]:
mediana_sobres = np.median(sobres_sim)

print(f"Mediana de sobres necesarios (Etapa 1)      : {mediana_sobres:.1f}")
print(f"Primer M con P ≥ 50% (Etapa 2)              : {M50}")
print()
print("Comparación:")
print(f"  Ambos valores son cercanos porque la mediana de la distribución")
print(f"  del número de sobres es el valor M tal que P(completar ≤ M) ≈ 0.5,")
print(f"  que es exactamente lo que mide la Etapa 2.")
print(f"  Diferencia: {abs(mediana_sobres - M50):.1f} sobres.")

### Pregunta 3: Cota superior para P(falta al menos una estampa) y comparación con M = 50

In [ ]:
# Por unión de eventos:
#   P(falta ≥1 estampa) ≤ sum_{i=1}^{N} P(estampa i no obtenida en M sobres)
#                        = N * P(estampa específica no obtenida)
#
# P(estampa i no está en 1 sobre) = 1 - S/N  (elegir S de N sin reemplazo)
# P(estampa i no está en M sobres) = (1 - S/N)^M   [sobres independientes]
#
# El enunciado usa la aproximación exponencial: e^(-MS/N)

M_test = 50

# Cota exacta (usando probabilidad exacta por sobre)
p_no_en_sobre = 1 - S/N
cota_exacta   = N * (p_no_en_sobre ** M_test)

# Aproximación exponencial del enunciado
cota_exp      = N * np.exp(-M_test * S / N)

# Probabilidad de NO completar = 1 - P(completar)
p_fallo_sim   = 1 - prob_exito.get(M_test, np.nan)

print(f"Para M = {M_test} sobres, N = {N}, S = {S}:")
print()
print(f"  Cota superior exacta (unión):      {cota_exacta:.6f}")
print(f"  Cota superior exp e^(-MS/N)·N:     {cota_exp:.6f}")
print(f"  P(NO completar) simulada:          {p_fallo_sim:.6f}")
print(f"  P(completar)    simulada:          {prob_exito.get(M_test, np.nan):.4f}")
print()
print("Interpretación:")
print(f"  La cota exponencial (~{cota_exp:.4f}) está por encima del valor simulado")
print(f"  ({p_fallo_sim:.4f}), lo que confirma que es una cota SUPERIOR válida.")
if cota_exp > 1:
    print(f"  Sin embargo, la cota > 1, lo que la hace trivial (no informativa) para este M.")
else:
    print(f"  La cota es {'útil (ajustada)' if cota_exp < 2*p_fallo_sim else 'conservadora (muy holgada)'}.")